<a href="https://colab.research.google.com/github/icosrle31/ENNOH/blob/main/Reading_NTCs_Offshore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction

In [ ]:
! pip install pypsa highspy openpyxl "xarray<=2024.9.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 369.0/369.0 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.8/208.8 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 120.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 100.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 85.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 65.8 MB/s eta 0:00:00
  Attempting uninstall: xarray
    Found existing installation: xarray 2025.12.0
    Uninstalling xarray-2025.12.0:
      Successfully uninstalled xarray-2025.12.0


In [ ]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

Mounted at /content/drive


In [ ]:
# Import packages
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import openpyxl
import json
import time
import pypsa
import warnings

ROOT_DIR = os.getcwd()
PROJECT_DIR = os.path.join(ROOT_DIR, "drive/MyDrive/Colab_Notebooks/ENNOH/Zonal_model")


In [ ]:
sys.path.append(PROJECT_DIR)

from modules.getting_input_data import get_input_data

input_file_name = "input_file.xlsx"
input_data = get_input_data(PROJECT_DIR, input_file_name)

File found at: /content/drive/MyDrive/Colab_Notebooks/ENNOH/Zonal_model/input_file.xlsx
The input data has been imported.


In [ ]:
input_data

{'project_name': 'Europe',
 'regions': '["Albania", "Austria", "Belgium", "Bosnia and Herzegovina", "Bulgaria", "Croatia", "Cyprus", "Czechia", "Denmark", "Estonia", "Finland", "France", "Germany", "Greece", "Hungary", "Ireland", "Italy", "Latvia", "Lithuania", "Luxembourg", "Malta", "Montenegro", "Netherlands", "North Macedonia", "Norway", "Poland", "Portugal", "Romania", "Serbia", "Slovakia", "Slovenia", "Spain", "Sweden", "Switzerland", "United Kingdom"]',
 'zones': '["AL00", "AT00", "BA00", "BE00", "BG00", "CH00", "CY00", "CZ00", "DE00", "DKE1", "DKW1", "EE00", "ES00", "FI00", "FR00", "GR00", "GR03", "HR00", "HU00", "IE00", "ITCA", "ITCN", "ITCS", "ITN1", "ITS1", "ITSA", "ITSI", "LT00", "LUF1", "LUG1", "LUV1", "LV00", "MD00", "ME00", "MK00", "MT00", "NL00", "NOM1", "NON1", "NOS1", "NOS2", "NOS3", "PL00", "PT00", "RO00", "RS00", "SE01", "SE02", "SE03", "SE04", "SI00", "SK00", "TR00", "UA00", "UK00", "UKNI"]',
 'data_set': 'TYNDP_scenario_2026',
 'year': 2035,
 'scenario': 'DE',
 'we

In [ ]:
zones = json.loads(input_data["zones"])
year = input_data["year"]
scenario =  input_data["scenario"]
project_name =  input_data["project_name"]

DATA_DIR = os.path.join(PROJECT_DIR, str(input_data["data_set"]),input_data["raw_data_dir"])

INTER_DIR = os.path.join(PROJECT_DIR, str(input_data["data_set"]),input_data["inter_dir"],input_data["project_name"], input_data["scenario"], str(input_data["year"]))
INTER_DIR_PROFILE = os.path.join(PROJECT_DIR, str(input_data["data_set"]), input_data['inter_dir'], input_data["project_name"], input_data["scenario"], str(input_data["year"]), f'profile_{input_data["weather_profile"]}')

MODEL_DATA_DIR = os.path.join(PROJECT_DIR,str(input_data["data_set"]),input_data['model_data_dir'],input_data["project_name"],input_data["scenario"], str(input_data["year"]))
MODEL_DATA_FIX = os.path.join(PROJECT_DIR,str(input_data["data_set"]),input_data['model_data_dir'])



# Net Transfer Capacity - Electricity

## Total NTCs

In [ ]:
import pandas as pd
import os

# Path to the specific line data file
elec_line_path = os.path.join(DATA_DIR, "Line-data", "ReferenceGrid_Electricity.xlsx")

try:
    # Read the specific sheet for the target year (e.g., 'Year_2035')
    target_sheet = f'Year_{year}'
    ntc_raw_df = pd.read_excel(elec_line_path, sheet_name=target_sheet)
    print(f"✅ Successfully loaded ReferenceGrid_Electricity for sheet {target_sheet}.")

    ntc_prepared = []

    if len(zones) == 1:
        print(f"ℹ️ Skipping electricity NTC processing: Only one zone ({zones[0]}) detected.")
    else:
        for _, row in ntc_raw_df.iterrows():
            if '-' in str(row['Border']):
                z1, z2 = str(row['Border']).split('-')
                z1 = z1.strip()
                z2 = z2.strip()

                cap1 = float(row.get('Summary Direction 1', 0))
                cap2 = float(row.get('Summary Direction 2', 0))

                # Direction 1: z1 -> z2
                ntc_prepared.append({
                    'zone 1': z1,
                    'zone 2': z2,
                    'capacity, MW': cap1,
                    'losses': 0.95
                })

                # Direction 2: z2 -> z1
                ntc_prepared.append({
                    'zone 1': z2,
                    'zone 2': z1,
                    'capacity, MW': cap2,
                    'losses': 0.95
                })

    NTC_tot_df = pd.DataFrame(ntc_prepared)

    # Ensure NTC_tot_df always has the expected columns, even if empty
    if NTC_tot_df.empty:
        NTC_tot_df = pd.DataFrame(columns=['zone 1', 'zone 2', 'capacity, MW', 'losses'])
        print("☐ No matching connections found for the current project zones.")
    else:
        print(f"Found {len(NTC_tot_df)} total directed connections.")

except Exception as e:
    print(f"❌ An error occurred: {e}")
    NTC_tot_df = pd.DataFrame(columns=['zone 1', 'zone 2', 'capacity, MW', 'losses'])

✅ Successfully loaded ReferenceGrid_Electricity for sheet Year_2035.
Found 386 total directed connections.


## Internal NTCs

In [ ]:
if 'NTC_tot_df' in locals() and 'zones' in locals():
    # Filter NTCs where both nodes are in the project's predefined zones
    NTC_elec_df = NTC_tot_df[(NTC_tot_df['zone 1'].isin(zones)) & (NTC_tot_df['zone 2'].isin(zones))].copy()
    print(f"Found {len(NTC_elec_df)} strictly internal electricity connections.")
else:
    print("❌ Either NTC_tot_df or zones is not defined.")

Found 226 strictly internal electricity connections.


In [ ]:
NTC_elec_df

,zone 1,zone 2,"capacity, MW",losses
0,AL00,GR00,610.0,0.95
1,GR00,AL00,1040.0,0.95
2,AL00,ME00,300.0,0.95
3,ME00,AL00,300.0,0.95
4,AL00,MK00,298.0,0.95
...,...,...,...,...
279,UA00,RO00,50.0,0.95
280,SK00,UA00,400.0,0.95
281,UA00,SK00,400.0,0.95
282,MD00,UA00,600.0,0.95


## Offshore (Direct) NTCs

In [ ]:
if 'NTC_tot_df' in locals():
    # Filter NTC_tot_df where exactly one zone is offshore to get project-relevant direct offshore links
    # This matches the full structure: zone 1, zone 2, capacity, MW, losses
    offshore_direct_project_df = NTC_tot_df[
        ((NTC_tot_df['zone 1'].isin(zones)) & (NTC_tot_df['zone 2'].str.contains('OFF', na=False))) |
        ((NTC_tot_df['zone 2'].isin(zones)) & (NTC_tot_df['zone 1'].str.contains('OFF', na=False)))
    ].copy()

    # Update the summary list for backward compatibility if needed
    NTC_OFF_s = list(zip(offshore_direct_project_df['zone 1'], offshore_direct_project_df['zone 2']))

    print(f"Found {len(offshore_direct_project_df)} Project-specific Offshore Direct NTC connections.")
    display(offshore_direct_project_df)
else:
    print("❌ NTC_tot_df is not available.")

Found 84 Project-specific Offshore Direct NTC connections.


,zone 1,zone 2,"capacity, MW",losses
284,BE00,BEO1_OFF,1400.0,0.95
285,BEO1_OFF,BE00,1400.0,0.95
286,BE00,BEO2_OFF,0.0,0.95
287,BEO2_OFF,BE00,0.0,0.95
288,BE00,DKNS_OFF,0.0,0.95
...,...,...,...,...
381,NOM1,NONC_OFF,0.0,0.95
382,NOND_OFF,NON1,0.0,0.95
383,NON1,NOND_OFF,0.0,0.95
384,NOWB_OFF,NOS3,0.0,0.95


In [ ]:
if 'offshore_direct_project_df' in locals():
    # Keep only connections where zone 1 is offshore (contains 'OFF')
    # AND zone 2 is an onshore project zone (does NOT contain 'OFF')
    offshore_direct_project_df = offshore_direct_project_df[
        (offshore_direct_project_df['zone 1'].str.contains('OFF', na=False)) &
        (~offshore_direct_project_df['zone 2'].str.contains('OFF', na=False)) &
        (offshore_direct_project_df['zone 2'].isin(zones))
    ].copy()

    print(f"Updated offshore_direct_project_df to {len(offshore_direct_project_df)} connections (strictly offshore to onshore).")
    display(offshore_direct_project_df.head())

Updated offshore_direct_project_df to 42 connections (strictly offshore to onshore).


,zone 1,zone 2,"capacity, MW",losses
285,BEO1_OFF,BE00,1400.0,0.95
287,BEO2_OFF,BE00,0.0,0.95
289,DKNS_OFF,BE00,0.0,0.95
290,BEO1_OFF,UK00,1400.0,0.95
293,DEKF_OFF,DE00,400.0,0.95


In [ ]:
len(offshore_direct_project_df)

42

## Offshore-Offshore NTCs

In [ ]:
import pandas as pd

if 'NTC_tot_df' in locals():
    # Create a new DataFrame by filtering NTC_tot_df
    # to include connections where both 'zone 1' and 'zone 2' contain 'OFF'
    offshore_offshore_df = NTC_tot_df[
        (NTC_tot_df['zone 1'].str.contains('OFF', na=False)) &
        (NTC_tot_df['zone 2'].str.contains('OFF', na=False))
    ].copy()

    print(f"✅ Successfully created offshore_offshore_df with {len(offshore_offshore_df)} connections where both zones are offshore.")
else:
    print("❌ NTC_tot_df is not available. Please ensure the cell loading 'ReferenceGrid_Electricity.xlsx' was run.")

✅ Successfully created offshore_offshore_df with 18 connections where both zones are offshore.


In [ ]:
if 'offshore_offshore_df' in locals():
    # Combine 'zone 1' and 'zone 2' columns and find unique values
    offshore_zones_list = sorted(list(set(offshore_offshore_df['zone 1'].unique()) | set(offshore_offshore_df['zone 2'].unique())))

    print(f"Found {len(offshore_zones_list)} unique zones in offshore_offshore_df:")
    print(offshore_zones_list)
else:
    print("❌ offshore_offshore_df is not defined. Please ensure the cell creating it has been executed.")

Found 18 unique zones in offshore_offshore_df:
['DEKF_OFF', 'DKB2_OFF', 'DKBH_OFF', 'DKKF_OFF', 'NL0C_OFF', 'NL0D_OFF', 'NL0E_OFF', 'NL0G_OFF', 'NL0J_OFF', 'NL0K_OFF', 'NL0L_OFF', 'NL0M_OFF', 'NL0P_OFF', 'NL0Q_OFF', 'NL0S_OFF', 'NL0T_OFF', 'NL0X_OFF', 'NL0Y_OFF']


## Offshore Hub Definition

In [ ]:
offshore_hubs = {
    "BEO1_OFF": ["BE00", "UK00"],
    "NLLL_OFF": ["NL00", "UK00"],
    "NOSF_OFF": ["DE00", "NOS2"],
    "DKBH_OFF": ["DE00", "DKE1", "DKB2_OFF"],
    "DKNS_OFF": ["BE00", "DE00", "DKW1"],
    "DEKF_OFF": ["DE00", "DKKF_OFF"],
}


In [ ]:
import os
import pandas as pd

# 1. Locate and Load the Offshore Profiles
# Based on previous attempts, I'll use the INTER_DIR to find the correct profiles file
profiles_filename = 'Offshore_profiles.csv'
profiles_path = os.path.join(INTER_DIR_PROFILE, profiles_filename)

try:
    offshore_profiles_df = pd.read_csv(profiles_path, index_col=0)
    profile_list = set(offshore_profiles_df.columns)
    print(f"✅ Successfully loaded {len(profile_list)} profiles from {profiles_filename}.")

    # 2. Extract offshore zones from the network dataframes
    # We extract zones containing 'OFF' from both direct and offshore-offshore dataframes
    network_offshore_zones = set()

    if 'offshore_direct_project_df' in locals():
        z1 = set(offshore_direct_project_df['zone 1'][offshore_direct_project_df['zone 1'].str.contains('OFF', na=False)])
        z2 = set(offshore_direct_project_df['zone 2'][offshore_direct_project_df['zone 2'].str.contains('OFF', na=False)])
        network_offshore_zones.update(z1 | z2)

    if 'offshore_offshore_df' in locals():
        z1_off = set(offshore_offshore_df['zone 1'])
        z2_off = set(offshore_offshore_df['zone 2'])
        network_offshore_zones.update(z1_off | z2_off)

    print(f"Found {len(network_offshore_zones)} offshore zones in network topology.")

    # 3. Validation Check
    missing_profiles = network_offshore_zones - profile_list

    if not missing_profiles:
        print("✅ Success: All offshore zones in the network have corresponding profiles.")
    else:
        print(f"❌ Warning: {len(missing_profiles)} zones are missing profiles!")
        print(f"Missing: {sorted(list(missing_profiles))}")

except FileNotFoundError:
    print(f"❌ Could not find {profiles_filename} at {profiles_path}.")
except Exception as e:
    print(f"❌ An error occurred: {e}")

✅ Successfully loaded 54 profiles from Offshore_profiles.csv.
Found 44 offshore zones in network topology.
✅ Success: All offshore zones in the network have corresponding profiles.


### Delete UNUSED Offshore profiles

In [ ]:
import pandas as pd

# 1. Gather all offshore nodes currently mentioned in the network topology
used_offshore_nodes = set()

if 'offshore_direct_project_df' in locals():
    used_offshore_nodes.update(offshore_direct_project_df['zone 1'].unique())
    used_offshore_nodes.update(offshore_direct_project_df['zone 2'].unique())

if 'offshore_offshore_df' in locals():
    used_offshore_nodes.update(offshore_offshore_df['zone 1'].unique())
    used_offshore_nodes.update(offshore_offshore_df['zone 2'].unique())

if 'offshore_hubs_project' in locals():
    used_offshore_nodes.update(offshore_hubs_project.keys())

# Keep only those that contain 'OFF'
used_offshore_nodes = {n for n in used_offshore_nodes if 'OFF' in str(n)}

# 2. Get the list of all available offshore profiles
if 'offshore_profiles_df' in locals():
    all_profile_nodes = set(offshore_profiles_df.columns)

    # 3. Find profiles that are NOT in the used network nodes
    unused_offshore_profiles = sorted(list(all_profile_nodes - used_offshore_nodes))

    print(f"Found {len(unused_offshore_profiles)} offshore profiles not mentioned in the current network.")
    print("\nTop 10 Unused Offshore Nodes:")
    display(unused_offshore_profiles[:10])
else:
    print("❌ offshore_profiles_df is not available. Please ensure the profile loading cell was run.")

Found 10 offshore profiles not mentioned in the current network.

Top 10 Unused Offshore Nodes:


['DKBF_OFF',
 'DKN6_OFF',
 'DKN7_OFF',
 'DKN8_OFF',
 'DKN9_OFF',
 'ESA1_OFF',
 'ESA2_OFF',
 'ESAS_OFF',
 'ESG1_OFF',
 'NL0R_OFF']

In [ ]:
if 'offshore_profiles_df' in locals() and 'unused_offshore_profiles' in locals():
    # Drop the identified 10 unused columns
    offshore_profiles_df = offshore_profiles_df.drop(columns=unused_offshore_profiles)

    print(f"✅ Successfully deleted {len(unused_offshore_profiles)} unused profiles.")
    print(f"New shape of offshore_profiles_df: {offshore_profiles_df.shape}")
    print(f"Total profiles remaining: {len(offshore_profiles_df.columns)}")
else:
    print("❌ offshore_profiles_df or unused_offshore_profiles list not found.")

✅ Successfully deleted 10 unused profiles.
New shape of offshore_profiles_df: (8760, 44)
Total profiles remaining: 44


## Virtual zones and NTCs related to that

In [ ]:
virtual_zones = ['ITCO', 'ITVI', 'PL00I', 'PL00E', 'LUB1']

In [ ]:
connection_virtual = {
    'Europe' : ['ITCO', 'ITVI', 'PL00I', 'PL00E', 'LUB1'],
    'Italy' : ['ITCO', 'ITVI'],
    'Poland' : ['PL00I', 'PL00E'],
    'Luxemburg' : ['LUB1']
}

In [ ]:
if 'NTC_tot_df' in locals() and 'virtual_zones' in locals() and 'zones' in locals():
    # Identify all NTCs involving any virtual zone
    virtual_ntc_df = NTC_tot_df[
        NTC_tot_df['zone 1'].isin(virtual_zones) |
        NTC_tot_df['zone 2'].isin(virtual_zones)
    ].copy()

    # Filter to find ONLY those relevant to the project zones (Italy, Austria, Slovenia)
    # A virtual NTC is project-relevant if one side is a project zone and the other is virtual
    virtual_ntc_df_proj = virtual_ntc_df[
        (virtual_ntc_df['zone 1'].isin(zones) | virtual_ntc_df['zone 2'].isin(zones))
    ].copy()

else:
    print("❌ NTC_tot_df, virtual_zones, or zones is not available.")
    virtual_ntc_df_proj = pd.DataFrame(columns=['zone 1', 'zone 2', 'capacity, MW', 'losses'])


In [ ]:
virtual_ntc_df_proj

,zone 1,zone 2,"capacity, MW",losses
32,BE00,LUB1,380.0,0.95
33,LUB1,BE00,0.0,0.95
64,CZ00,PL00I,600.0,0.95
65,PL00I,CZ00,0.0,0.95
84,DE00,PL00I,2000.0,0.95
85,PL00I,DE00,0.0,0.95
174,ITCN,ITCO,400.0,0.95
175,ITCO,ITCN,400.0,0.95
180,ITCO,ITSA,425.0,0.95
181,ITSA,ITCO,500.0,0.95


## Export and Import NTCs

In [ ]:
import pandas as pd

import_NTCs = []
export_NTCs = []

if 'ntc_raw_df' in locals():
    for _, row in ntc_raw_df.iterrows():
        border = str(row['Border']).strip()
        if '-' in border:
            z1, z2 = border.split('-')
            z1 = z1.strip()
            z2 = z2.strip()

            cap1 = float(row.get('Summary Direction 1', 0))
            cap2 = float(row.get('Summary Direction 2', 0))
            loss_val = row.get('Losses', 0.95)

            if isinstance(loss_val, str):
                loss_val = float(loss_val.replace(',', '.'))

            # NEW: Exclude offshore zones to ensure mutually exclusive categories
            if 'OFF' in z1 or 'OFF' in z2:
                continue

            # First zone in project -> Export
            if z1 in zones and z2 not in zones and z2 not in virtual_zones:
                export_NTCs.append({'zone 1': z1, 'zone 2': z2, 'capacity, MW': cap1, 'losses': loss_val})
                import_NTCs.append({'zone 1': z2, 'zone 2': z1, 'capacity, MW': cap2, 'losses': loss_val})

            # Second zone in project -> Import
            elif z1 not in zones and z2 in zones and z1 not in virtual_zones:
                import_NTCs.append({'zone 1': z1, 'zone 2': z2, 'capacity, MW': cap1, 'losses': loss_val})
                export_NTCs.append({'zone 1': z2, 'zone 2': z1, 'capacity, MW': cap2, 'losses': loss_val})

    import_NTCs_df = pd.DataFrame(import_NTCs)
    export_NTCs_df = pd.DataFrame(export_NTCs)

    print(f"Created import_NTCs_df with {len(import_NTCs_df)} connections.")
    print(f"Created export_NTCs_df with {len(export_NTCs_df)} connections.")

else:
    print("❌ ntc_raw_df is not available.")

Created import_NTCs_df with 9 connections.
Created export_NTCs_df with 9 connections.


## External_external NTCs

In [ ]:
if 'import_NTCs_df' in locals() and 'export_NTCs_df' in locals():
    # In import_NTCs_df, the external zone is 'zone 1'
    external_import_zones = set(import_NTCs_df['zone 1'].unique())

    # In export_NTCs_df, the external zone is 'zone 2'
    external_export_zones = set(export_NTCs_df['zone 2'].unique())

    # Combine both sets to get all unique external zones
    all_external_zones = sorted(list(external_import_zones | external_export_zones))

    print(f"Found {len(all_external_zones)} unique external zones in import/export:")
    print(all_external_zones)
else:
    print("❌ import_NTCs_df or export_NTCs_df is not available.")

Found 6 unique external zones in import/export:
['EG00', 'IL00', 'IS00', 'LY00', 'MA00', 'TN00']


In [ ]:
if 'NTC_tot_df' in locals() and 'zones' in locals() and 'virtual_zones' in locals():
    external_external_list = []

    # Iterate through all NTCs to identify external-to-external connections
    for _, row in NTC_tot_df.iterrows():
        z1 = row['zone 1']
        z2 = row['zone 2']

        # A connection is external-external if both zones are NOT in the project's zones
        # AND neither zone is an 'OFF'shore zone (these are handled separately).
        # AND neither zone is a 'virtual' zone to avoid overlap
        if (z1 not in zones and z2 not in zones and
            'OFF' not in z1 and 'OFF' not in z2 and
            z1 not in virtual_zones and z2 not in virtual_zones):
            external_external_list.append((z1, z2))

    # Remove duplicates from the list of external-external NTCs
    external_external_NTCs = list(dict.fromkeys(external_external_list))

    # Create a DataFrame from the identified external-external connections
    external_external_df = pd.DataFrame(external_external_NTCs, columns=['zone 1', 'zone 2'])

    print(f"✅ Successfully created external_external_df with {len(external_external_df)} connections.")
    if not external_external_df.empty:
        print("Sample External-to-External Connections:")
else:
    print("❌ NTC_tot_df, zones, or virtual_zones is not available. Cannot create external_external_df.")

✅ Successfully created external_external_df with 10 connections.
Sample External-to-External Connections:


## NTCs Summary

In [ ]:
import pandas as pd

# Create sets of tuples (zone 1, zone 2) for each category to easily find overlaps
set_tot = set(zip(NTC_tot_df['zone 1'], NTC_tot_df['zone 2'])) if 'NTC_tot_df' in locals() else set()
set_int = set(zip(NTC_elec_df['zone 1'], NTC_elec_df['zone 2'])) if 'NTC_elec_df' in locals() else set()
set_exp = set(zip(export_NTCs_df['zone 1'], export_NTCs_df['zone 2'])) if 'export_NTCs_df' in locals() else set()
set_imp = set(zip(import_NTCs_df['zone 1'], import_NTCs_df['zone 2'])) if 'import_NTCs_df' in locals() else set()
set_vir = set(zip(virtual_ntc_df['zone 1'], virtual_ntc_df['zone 2'])) if 'virtual_ntc_df' in locals() else set()

# Corrected: Use offshore_direct_project_df instead of undefined Offshore_NTCs
set_off_direct = set(zip(offshore_direct_project_df['zone 1'], offshore_direct_project_df['zone 2'])) if 'offshore_direct_project_df' in locals() else set()
set_off_offshore = set(tuple(row) for row in offshore_offshore_df[['zone 1', 'zone 2']].values) if 'offshore_offshore_df' in locals() else set()

# External-to-External category
set_ext_ext = set(tuple(row) for row in external_external_df[['zone 1', 'zone 2']].values) if 'external_external_df' in locals() else set()

print(f"Total Unique NTCs (NTC_tot_df): {len(set_tot)}")
print(f"  - Internal: {len(set_int)}")
print(f"  - Export: {len(set_exp)}")
print(f"  - Import: {len(set_imp)}")
print(f"  - Offshore (direct): {len(set_off_direct)}")
print(f"  - Offshore-Offshore: {len(set_off_offshore)}")
print(f"  - External-to-External: {len(set_ext_ext)}")
print(f"  - Virtual: {len(set_vir)}")

all_categorized = set_int | set_exp | set_imp | set_off_direct | set_off_offshore | set_ext_ext | set_vir
unclassified = set_tot - all_categorized

print(f"\nSum of parts: {len(set_int) + len(set_exp) + len(set_imp) + len(set_off_direct) + len(set_off_offshore) + len(set_ext_ext) + len(set_vir)}")
print(f"Number of unclassified connections: {len(unclassified)}")

if unclassified:
    print("Unclassified connections sample:")
    for conn in list(unclassified)[:5]:
        print(f"  - {conn}")
else:
    print("✅ All electricity connections are successfully categorized.")

Total Unique NTCs (NTC_tot_df): 386
  - Internal: 226
  - Export: 9
  - Import: 9
  - Offshore (direct): 42
  - Offshore-Offshore: 18
  - External-to-External: 10
  - Virtual: 30

Sum of parts: 344
Number of unclassified connections: 42
Unclassified connections sample:
  - ('DKE1', 'DKK2_OFF')
  - ('UK00', 'NLLL_OFF')
  - ('DKW1', 'DKNS_OFF')
  - ('DKW1', 'DKN2_OFF')
  - ('DE00', 'DKBH_OFF')


# Project NTCs - Filter

In [ ]:
import pandas as pd

# Get the virtual zones specifically linked to the current project (e.g., 'Italy')
project_specific_virtual_zones = connection_virtual.get(project_name, [])

# Ensure virtual_ntc_df, project_specific_virtual_zones, and zones are available
if 'virtual_ntc_df' in locals() and project_specific_virtual_zones and 'zones' in locals():
    # Filter NTCs where one zone is a project-specific virtual zone
    # AND the other zone is one of the project's main zones.
    virtual_NTC_project_df = virtual_ntc_df[
        (
            (virtual_ntc_df['zone 1'].isin(project_specific_virtual_zones)) &
            (virtual_ntc_df['zone 2'].isin(zones))
        ) | (
            (virtual_ntc_df['zone 2'].isin(project_specific_virtual_zones)) &
            (virtual_ntc_df['zone 1'].isin(zones))
        )
    ].copy()

    print(f"✅ Successfully created virtual_NTC_project_df with {len(virtual_NTC_project_df)} project-specific virtual connections.")
    display(virtual_NTC_project_df)
else:
    print("❌ Required variables (virtual_ntc_df, project_specific_virtual_zones, or zones) are not available.")
    virtual_NTC_project_df = pd.DataFrame(columns=['zone 1', 'zone 2', 'capacity, MW', 'losses'])

✅ Successfully created virtual_NTC_project_df with 28 project-specific virtual connections.


,zone 1,zone 2,"capacity, MW",losses
32,BE00,LUB1,380.0,0.95
33,LUB1,BE00,0.0,0.95
64,CZ00,PL00I,600.0,0.95
65,PL00I,CZ00,0.0,0.95
84,DE00,PL00I,2000.0,0.95
85,PL00I,DE00,0.0,0.95
174,ITCN,ITCO,400.0,0.95
175,ITCO,ITCN,400.0,0.95
180,ITCO,ITSA,425.0,0.95
181,ITSA,ITCO,500.0,0.95


In [ ]:
import pandas as pd

# 1. Project-Specific Offshore (Direct) NTCs
# Filters NTCs where one side is a project zone and the other is an offshore node
# (Using the previously calculated Offshore_NTCs list of tuples)
offshore_direct_project_df = NTC_tot_df[
    (NTC_tot_df['zone 1'].isin(zones) & NTC_tot_df['zone 2'].str.contains('OFF', na=False)) |
    (NTC_tot_df['zone 2'].isin(zones) & NTC_tot_df['zone 1'].str.contains('OFF', na=False))
].copy()

# 2. Project-Specific Offshore-Offshore NTCs
# Get offshore nodes that are directly connected to our project zones first
project_direct_offshore_nodes = set(offshore_direct_project_df['zone 1'].unique()) | set(offshore_direct_project_df['zone 2'].unique())
project_direct_offshore_nodes = {node for node in project_direct_offshore_nodes if 'OFF' in node}

# Filter the global offshore-offshore connections to only those involving these nodes
offshore_offshore_project_df = offshore_offshore_df[
    offshore_offshore_df['zone 1'].isin(project_direct_offshore_nodes) |
    offshore_offshore_df['zone 2'].isin(project_direct_offshore_nodes)
].copy()

print(f"--- Project-Specific Offshore NTCs: {project_name} ---")
print(f"- Offshore Direct: {len(offshore_direct_project_df)}")
print(f"- Offshore-Offshore: {len(offshore_offshore_project_df)}")

if not offshore_direct_project_df.empty:
    display(offshore_direct_project_df.head())
else:
    print("ℹ️ No direct offshore connections found for this project.")

--- Project-Specific Offshore NTCs: Europe ---
- Offshore Direct: 84
- Offshore-Offshore: 18


,zone 1,zone 2,"capacity, MW",losses
284,BE00,BEO1_OFF,1400.0,0.95
285,BEO1_OFF,BE00,1400.0,0.95
286,BE00,BEO2_OFF,0.0,0.95
287,BEO2_OFF,BE00,0.0,0.95
288,BE00,DKNS_OFF,0.0,0.95


In [ ]:
import pandas as pd

# 1. Project-Specific Internal NTCs
NTC_elec_project_df = NTC_elec_df[NTC_elec_df['zone 1'].isin(zones) & NTC_elec_df['zone 2'].isin(zones)].copy()

# 2. Project-Specific Import NTCs (External -> Project Zone)
import_NTCs_project_df = import_NTCs_df[import_NTCs_df['zone 2'].isin(zones)].copy()

# 3. Project-Specific Export NTCs (Project Zone -> External)
export_NTCs_project_df = export_NTCs_df[export_NTCs_df['zone 1'].isin(zones)].copy()

# 4. Project-Specific External-External NTCs (Connections between external neighbors of the project)
# Get neighbors of project zones
neighbors = set(import_NTCs_project_df['zone 1'].unique()) | set(export_NTCs_project_df['zone 2'].unique())
external_external_project_df = external_external_df[
    external_external_df['zone 1'].isin(neighbors) &
    external_external_df['zone 2'].isin(neighbors)
].copy()

print(f"--- Project-Specific Dataframes Created for: {project_name} ---")
print(f"- Internal: {len(NTC_elec_project_df)}")
print(f"- Import:   {len(import_NTCs_project_df)}")
print(f"- Export:   {len(export_NTCs_project_df)}")
print(f"- Ext-Ext:  {len(external_external_project_df)}")
print(f"- Virtual:  {len(virtual_NTC_project_df)}")

--- Project-Specific Dataframes Created for: Europe ---
- Internal: 226
- Import:   9
- Export:   9
- Ext-Ext:  4
- Virtual:  28


In [ ]:
offshore_hubs_project = {}

# Check if any offshore hub is connected to the project's zones
for hub, onshore_nodes in offshore_hubs.items():
    # Check for intersection between the hub's nodes and project zones
    relevant_onshore = [node for node in onshore_nodes if node in zones]

    if relevant_onshore:
        offshore_hubs_project[hub] = relevant_onshore

if offshore_hubs_project:
    print(f"Found {len(offshore_hubs_project)} project-relevant offshore hubs:")
    for hub, nodes in offshore_hubs_project.items():
        print(f"  - {hub} connected to: {nodes}")
else:
    print("ℹ️ No offshore hubs from the definition are connected to the current project zones.")

Found 6 project-relevant offshore hubs:
  - BEO1_OFF connected to: ['BE00', 'UK00']
  - NLLL_OFF connected to: ['NL00', 'UK00']
  - NOSF_OFF connected to: ['DE00', 'NOS2']
  - DKBH_OFF connected to: ['DE00', 'DKE1']
  - DKNS_OFF connected to: ['BE00', 'DE00', 'DKW1']
  - DEKF_OFF connected to: ['DE00']


In [ ]:
print(f"--- Project-Relevant NTC Summary: {project_name} ---")

# Calculate project-specific sets from the filtered dataframes
set_proj_int = set(zip(NTC_elec_project_df['zone 1'], NTC_elec_project_df['zone 2'])) if 'NTC_elec_project_df' in locals() else set()
set_proj_imp = set(zip(import_NTCs_project_df['zone 1'], import_NTCs_project_df['zone 2'])) if 'import_NTCs_project_df' in locals() else set()
set_proj_exp = set(zip(export_NTCs_project_df['zone 1'], export_NTCs_project_df['zone 2'])) if 'export_NTCs_project_df' in locals() else set()
set_proj_ext = set(zip(external_external_project_df['zone 1'], external_external_project_df['zone 2'])) if 'external_external_project_df' in locals() else set()
set_proj_vir = set(zip(virtual_NTC_project_df['zone 1'], virtual_NTC_project_df['zone 2'])) if 'virtual_NTC_project_df' in locals() else set()

# Total categorized relevant connections
set_proj_total_categorized = set_proj_int | set_proj_imp | set_proj_exp | set_proj_ext | set_proj_vir

# Verification against raw data filtered for project zones
raw_proj_filter = NTC_tot_df[NTC_tot_df['zone 1'].isin(zones) | NTC_tot_df['zone 2'].isin(zones)]
set_raw_proj = set(zip(raw_proj_filter['zone 1'], raw_proj_filter['zone 2']))

# Identify any project-specific connections that missed categorization
unclassified_proj = set_raw_proj - set_proj_total_categorized

print(f"Total Unique Project Connections: {len(set_raw_proj)}")
print(f"  - Internal:           {len(set_proj_int)}")
print(f"  - Import (to project): {len(set_proj_imp)}")
print(f"  - Export (from proj): {len(set_proj_exp)}")
print(f"  - External-to-External (Neighbors): {len(set_proj_ext)}")
print(f"  - Virtual (Project):  {len(set_proj_vir)}")
print(f"  - Offshore Hubs (Project): {len(offshore_hubs_project) if 'offshore_hubs_project' in locals() else 0}")
print(f"  - Unclassified Project NTCs: {len(unclassified_proj)}")

if unclassified_proj:
    print('\n\u2611\u1034 Missing Project Connections:')
    for conn in sorted(list(unclassified_proj)):
        print(f'    {conn}')
else:
    print(f'\n\u2705 100% of project-relevant connections for {project_name} are classified.')

--- Project-Relevant NTC Summary: Europe ---
Total Unique Project Connections: 356
  - Internal:           226
  - Import (to project): 9
  - Export (from proj): 9
  - External-to-External (Neighbors): 4
  - Virtual (Project):  28
  - Offshore Hubs (Project): 6
  - Unclassified Project NTCs: 84

☑ဴ Missing Project Connections:
    ('BE00', 'BEO1_OFF')
    ('BE00', 'BEO2_OFF')
    ('BE00', 'DKNS_OFF')
    ('BEO1_OFF', 'BE00')
    ('BEO1_OFF', 'UK00')
    ('BEO2_OFF', 'BE00')
    ('DE00', 'DEKF_OFF')
    ('DE00', 'DKBH_OFF')
    ('DE00', 'DKNS_OFF')
    ('DE00', 'NOSF_OFF')
    ('DEKF_OFF', 'DE00')
    ('DKBH_OFF', 'DE00')
    ('DKBH_OFF', 'DKE1')
    ('DKE1', 'DKBH_OFF')
    ('DKE1', 'DKHE_OFF')
    ('DKE1', 'DKK2_OFF')
    ('DKE1', 'DKKF_OFF')
    ('DKHE_OFF', 'DKE1')
    ('DKK2_OFF', 'DKE1')
    ('DKKA_OFF', 'DKW1')
    ('DKKF_OFF', 'DKE1')
    ('DKN1_OFF', 'DKW1')
    ('DKN2_OFF', 'DKW1')
    ('DKN3_OFF', 'DKW1')
    ('DKN4_OFF', 'DKW1')
    ('DKN5_OFF', 'DKW1')
    ('DKNS_OFF', 'BE0

In [ ]:
import pandas as pd

# 1. Reconstruct NTC_elec_dic with the specific categories requested
# Note: offshore_hubs_project was identified as empty in previous cells, but is included here for structure
NTC_elec_dic = {
    'internal': NTC_elec_project_df,
    'import': import_NTCs_project_df,
    'export': export_NTCs_project_df,
    'offshore_direct': offshore_direct_project_df,
    'offshore_offshore': offshore_offshore_project_df,
    'virtual': virtual_NTC_project_df,
    'ext_ext': external_external_project_df,
    'offshore_hub': offshore_hubs_project
}

print("--- NTC Electricity Dictionary Created ---")
for key, df_or_dict in NTC_elec_dic.items():
    count = len(df_or_dict) if isinstance(df_or_dict, (pd.DataFrame, list, dict)) else 0
    print(f"{key.replace('_', ' ').capitalize()}: {count} items")


--- NTC Electricity Dictionary Created ---
Internal: 226 items
Import: 9 items
Export: 9 items
Offshore direct: 84 items
Offshore offshore: 18 items
Virtual: 28 items
Ext ext: 4 items
Offshore hub: 6 items


In [ ]:
offshore_hubs_project

{'BEO1_OFF': ['BE00', 'UK00'],
 'NLLL_OFF': ['NL00', 'UK00'],
 'NOSF_OFF': ['DE00', 'NOS2'],
 'DKBH_OFF': ['DE00', 'DKE1'],
 'DKNS_OFF': ['BE00', 'DE00', 'DKW1'],
 'DEKF_OFF': ['DE00']}

# NTC Hydrogen

data is directly obtained from raw data folder

## Referent Total Network

In [ ]:
# hydrogen zones introduction
H2_zones = [
    'ALh2', 'ATh2', 'BAh2', 'BEh2', 'BGh2', 'CHh2', 'CYh2', 'CZh2', 'DEh2',
    'DEh2Z1', 'DKh2', 'EEh2', 'ESh2', 'FIh2', 'FIh2Al', 'FIh2N', 'FIh2S',
    'FRh2', 'FRh2N', 'FRh2S', 'FRh2SW', 'GRh2', 'HRh2', 'HUh2', 'IEh2',
    'ITh2', 'LTh2', 'LTh2Z1', 'LUh2', 'LVh2', 'MDh2', 'MKh2', 'MTh2',
    'NLh2', 'NOh2', 'PLh2', 'PTh2', 'PTh2Z1', 'ROh2', 'RSh2', 'SEh2',
    'SIh2', 'SKh2E', 'SKh2W', 'UKh2'
]

In [ ]:
{'DZ': ['ESh2', 'ITh2'],
 'IL': ['GRh2'],
 'MA': ['ESh2'],
 'NO': ['BEh2', 'DEh2', 'DKh2', 'FRh2', 'NLh2', 'UKh2'],
 'TN': ['ITh2'],
 'TR': ['GRh2', 'ITh2'],
 'UA': ['HUh2', 'PLh2', 'ROh2'],
 'Y_NO': ['DEh2']}

{'DZ': ['ESh2', 'ITh2'],
 'IL': ['GRh2'],
 'MA': ['ESh2'],
 'NO': ['BEh2', 'DEh2', 'DKh2', 'FRh2', 'NLh2', 'UKh2'],
 'TN': ['ITh2'],
 'TR': ['GRh2', 'ITh2'],
 'UA': ['HUh2', 'PLh2', 'ROh2'],
 'Y_NO': ['DEh2']}

In [ ]:
print(f"Total hydrogen zones in H2_zones: {len(H2_zones)}")


Total hydrogen zones in H2_zones: 45


In [ ]:
import pandas as pd
import os

# Path to the specific hydrogen line data file
h2_line_path = os.path.join(DATA_DIR, "Line-data", "ReferenceGrid_Hydrogen.xlsx")

try:
    target_sheet = f'Year_{year}'
    NTC_H2_tot_df = pd.read_excel(h2_line_path, sheet_name=target_sheet)

    # Set 'Border' as index
    if 'Border' in NTC_H2_tot_df.columns:
        NTC_H2_tot_df.set_index('Border', inplace=True)

    print(f"✅ Successfully loaded ReferenceGrid_Hydrogen for sheet {target_sheet}.")
    print(f"Total lines in Hydrogen NTC: {len(NTC_H2_tot_df)}")
    display(NTC_H2_tot_df.head())

except FileNotFoundError:
    print(f"❌ Error: The file {os.path.basename(h2_line_path)} was not found in {os.path.dirname(h2_line_path)}.")
except Exception as e:
    print(f"❌ An error occurred: {e}")

✅ Successfully loaded ReferenceGrid_Hydrogen for sheet Year_2035.
Total lines in Hydrogen NTC: 112


,Summary Direction 1,Summary Direction 2
Border,,
ALh2-HRh2,0.000000,0.000000
ATh2-CZh2,0.000000,0.000000
ATh2-DEh2,5296.610171,5296.610171
ATh2-HUh2,0.000000,0.000000
ATh2-IB_ITh2,4449.152542,5932.203392


### Directional dataframe

In [ ]:
import pandas as pd

NTC_H2_tot_dir_list = []

# Using the correctly loaded dataframe from cell CMY1dlKglEKW
if 'NTC_H2_tot_df' in locals():
    for border, row in NTC_H2_tot_df.iterrows():
        border_str = str(border).strip()
        if '-' in border_str:
            # Split nodes while preserving the full name (keeping 'h2' suffix if present)
            nodes = [n.strip() for n in border_str.split('-')]
            z1, z2 = nodes[0], nodes[1]

            # Direction 1: z1 -> z2
            NTC_H2_tot_dir_list.append({
                'Border': border_str,
                'zone 1': z1,
                'zone 2': z2,
                'capacity, MW': float(row.get('Summary Direction 1', 0)),
                'losses': 1.0
            })

            # Direction 2: z2 -> z1
            NTC_H2_tot_dir_list.append({
                'Border': border_str,
                'zone 1': z2,
                'zone 2': z1,
                'capacity, MW': float(row.get('Summary Direction 2', 0)),
                'losses': 1.0
            })

    # Create the global directed dataframe
    NTC_H2_tot_dir = pd.DataFrame(NTC_H2_tot_dir_list)

    print(f"✅ Expanded {len(NTC_H2_tot_df)} borders into {len(NTC_H2_tot_dir)} directed connections (NTC_H2_tot_dir).")
    display(NTC_H2_tot_dir.head())
else:
    print("❌ NTC_H2_tot_df not found. Please ensure cell CMY1dlKglEKW was executed.")

✅ Expanded 112 borders into 224 directed connections (NTC_H2_tot_dir).


,Border,zone 1,zone 2,"capacity, MW",losses
0,ALh2-HRh2,ALh2,HRh2,0.000000,1.0
1,ALh2-HRh2,HRh2,ALh2,0.000000,1.0
2,ATh2-CZh2,ATh2,CZh2,0.000000,1.0
3,ATh2-CZh2,CZh2,ATh2,0.000000,1.0
4,ATh2-DEh2,ATh2,DEh2,5296.610171,1.0


In [ ]:
import pandas as pd

# 1. Ensure we have the directed H2 dataframe
if 'NTC_H2_tot_dir' in locals():
    initial_count = len(NTC_H2_tot_dir)

    # 2. Filter out rows where Ammonia is the destination (zone 2)
    # This ensures connections are strictly Ammonia -> H2_Zone
    NTC_H2_tot_dir = NTC_H2_tot_dir[~NTC_H2_tot_dir['zone 2'].str.contains('Ammonia', case=False)].copy()

    removed_count = initial_count - len(NTC_H2_tot_dir)
    print(f"✅ Applied unidirectional logic to Ammonia connections.")
    print(f"Removed {removed_count} bidirectional/return records.")
    print(f"New total records in NTC_H2_tot_dir: {len(NTC_H2_tot_dir)}")

    # Display some Ammonia entries to verify unidirectionality
    display(NTC_H2_tot_dir[NTC_H2_tot_dir['zone 1'].str.contains('Ammonia', case=False)].head())
else:
    print("❌ NTC_H2_tot_dir not found. Please run the previous cell first.")

✅ Applied unidirectional logic to Ammonia connections.
Removed 13 bidirectional/return records.
New total records in NTC_H2_tot_dir: 211


,Border,zone 1,zone 2,"capacity, MW",losses
22,Ammonia_BE-BEh2,Ammonia_BE,BEh2,3788.841792,1.0
60,Ammonia_DE-DEh2,Ammonia_DE,DEh2,2390.663833,1.0
108,Ammonia_FRSW-FRh2SW,Ammonia_FRSW,FRh2SW,0.000000,1.0
116,Ammonia_FRW-FRh2,Ammonia_FRW,FRh2,0.000000,1.0
120,Ammonia_FRN-FRh2N,Ammonia_FRN,FRh2N,1694.915250,1.0


## Internal NTCs H2

In [ ]:
import pandas as pd

# Use the full H2 zone names for matching
h2_zones_set = set(H2_zones)

# Filter the directed dataframe (NTC_H2_tot_dir) where both zones are in the global H2 set
internal_mask = NTC_H2_tot_dir['zone 1'].isin(h2_zones_set) & NTC_H2_tot_dir['zone 2'].isin(h2_zones_set)
internal_NTC_H2 = NTC_H2_tot_dir[internal_mask].copy()

print(f"✅ Successfully processed {len(internal_NTC_H2)} internal directed connections.")

# Update the global audit sets to reflect this change
s_int = set(zip(internal_NTC_H2['zone 1'], internal_NTC_H2['zone 2']))


✅ Successfully processed 122 internal directed connections.


In [ ]:
import pandas as pd

# 1. Use full zone names for membership check
h2_zones_set = set(H2_zones)

# 2. Filter for ALL Internal H2 connections (Global)
internal_mask = NTC_H2_tot_dir['zone 1'].isin(h2_zones_set) & NTC_H2_tot_dir['zone 2'].isin(h2_zones_set)
internal_NTC_H2 = NTC_H2_tot_dir[internal_mask].copy()

# 3. Create specific filter for Z1 sub-zones
z1_mask = internal_NTC_H2['zone 1'].str.contains('Z1', na=False) | internal_NTC_H2['zone 2'].str.contains('Z1', na=False)
internal_Z1_NTC_H2 = internal_NTC_H2[z1_mask].copy()

print(f"✅ Total internal directed connections: {len(internal_NTC_H2)}")
print(f"✅ Sub-zone (Z1) internal connections: {len(internal_Z1_NTC_H2)}")

print("\nSample of Z1 Sub-zone Connections:")
display(internal_Z1_NTC_H2.head())

# Update audit set for consistency with previous cells
s_int = set(zip(internal_NTC_H2['zone 1'], internal_NTC_H2['zone 2']))


✅ Total internal directed connections: 122
✅ Sub-zone (Z1) internal connections: 6

Sample of Z1 Sub-zone Connections:


,Border,zone 1,zone 2,"capacity, MW",losses
214,DEh2-DEh2Z1,DEh2,DEh2Z1,416625.0,1.0
215,DEh2-DEh2Z1,DEh2Z1,DEh2,0.0,1.0
216,LTh2-LTh2Z1,LTh2,LTh2Z1,416625.0,1.0
217,LTh2-LTh2Z1,LTh2Z1,LTh2,0.0,1.0
218,PTh2-PTh2Z1,PTh2,PTh2Z1,416625.0,1.0


## Ammonia NTCs

In [ ]:
import pandas as pd

# FIXED: Use the directed dataframe NTC_H2_tot_dir which contains 'zone 1'
if 'NTC_H2_tot_dir' in locals():
    Ammonia_H2_df = NTC_H2_tot_dir[NTC_H2_tot_dir['zone 1'].str.contains('Ammonia', case=False, na=False)].copy()

    print(f"✅ Successfully created Ammonia_H2_df with {len(Ammonia_H2_df)} rows.")
    display(Ammonia_H2_df.head())
else:
    print("❌ NTC_H2_tot_dir not found. Please ensure previous hydrogen NTC processing cells were executed.")

✅ Successfully created Ammonia_H2_df with 13 rows.


,Border,zone 1,zone 2,"capacity, MW",losses
22,Ammonia_BE-BEh2,Ammonia_BE,BEh2,3788.841792,1.0
60,Ammonia_DE-DEh2,Ammonia_DE,DEh2,2390.663833,1.0
108,Ammonia_FRSW-FRh2SW,Ammonia_FRSW,FRh2SW,0.000000,1.0
116,Ammonia_FRW-FRh2,Ammonia_FRW,FRh2,0.000000,1.0
120,Ammonia_FRN-FRh2N,Ammonia_FRN,FRh2N,1694.915250,1.0


## Bottlenecks H2 NTCs

In [ ]:
# Define H2 Bottlenecks before they are used in filters
H2_Bottlenecks = ['IB_ITh2', 'IB_SKh2C', 'IB_SKh2E', 'IB_SKh2W', 'PLh2nbc','UKh2/INT', 'IB_GRh2P']

In [ ]:
H2_Bottlenecks_mapping = {
"ITN1" : ['IB_ITh2'],
"SK00" : ['IB_SKh2C', 'IB_SKh2E'],
"PL00" : ['PLh2nbc'],
"UK00" : ['UKh2/INT'],
"GR00" : ['IB_GRh2P']
}

In [ ]:
import pandas as pd

# 1. Define the list of bottleneck identifiers
bottlenecks = ['IB_ITh2', 'IB_SKh2C', 'IB_SKh2E', 'IB_SKh2W', 'PLh2nbc', 'UKh2/INT', 'IB_GRh2P']

# 2. Define the filtering function with exact matching support
def is_h2_bottleneck(row):
    z1, z2 = str(row['zone 1']), str(row['zone 2'])
    border = str(row.get('Border', ''))

    # Exclusion: No Ammonia nodes allowed
    if 'Ammonia' in z1 or 'Ammonia' in z2:
        return False

    # Inclusion: Check if the border or zones contain any of the bottleneck strings
    for b_id in bottlenecks:
        if b_id in border or b_id == z1 or b_id == z2:
            return True
    return False

# 3. Apply the filter to the DIRECTED dataframe
if 'NTC_H2_tot_dir' in locals():
    Bottlenecks_H2_df = NTC_H2_tot_dir[NTC_H2_tot_dir.apply(is_h2_bottleneck, axis=1)].copy()

    print(f"✅ Successfully filtered NTC_H2_tot_dir into Bottlenecks_H2_df.")
    print(f"Found {len(Bottlenecks_H2_df)} bottleneck connections.")
    display(Bottlenecks_H2_df.head())
else:
    print("❌ NTC_H2_tot_dir not found.")


✅ Successfully filtered NTC_H2_tot_dir into Bottlenecks_H2_df.
Found 38 bottleneck connections.


,Border,zone 1,zone 2,"capacity, MW",losses
8,ATh2-IB_ITh2,ATh2,IB_ITh2,4449.152542,1.0
9,ATh2-IB_ITh2,IB_ITh2,ATh2,5932.203392,1.0
30,BEh2-UKh2/INT,BEh2,UKh2/INT,0.000000,1.0
31,BEh2-UKh2/INT,UKh2/INT,BEh2,0.000000,1.0
42,CHh2-IB_ITh2,CHh2,IB_ITh2,0.000000,1.0


## Export and Import

In [ ]:
# external H2 zones
Ext_H2_zones = {
  'DZ'   : 'Algeria',
  'IL'   : 'Israel',
  'MA'   : 'Morocco',
  'Y_NO' : 'Norway',
  'NO'   : 'Norway',
  'TN'   : 'Tunisia',
  'TR'   : 'Turkey',
  'UA'   : 'Ukrania'
}

In [ ]:
# external hydrogen zones mapping to hydrogen zones
Ext_H2_mapping = {'DZ': ['ESh2', 'ITh2'],
 'IL': ['GRh2'],
 'MA': ['ESh2'],
 'NO': ['BEh2', 'DEh2', 'DKh2', 'FRh2', 'NLh2', 'UKh2'],
 'TN': ['ITh2'],
 'TR': ['GRh2', 'ITh2'],
 'UA': ['HUh2', 'PLh2', 'ROh2'],
 'Y_NO': ['DEh2']}

In [ ]:
import pandas as pd

if 'NTC_H2_tot_dir' in locals() and 'H2_zones' in globals() and 'Bottlenecks_H2_df' in globals():
    h2_zones_set = set(H2_zones)

    # Get exact bottleneck pairs to exclude them completely
    bottleneck_pairs = set(zip(Bottlenecks_H2_df['zone 1'], Bottlenecks_H2_df['zone 2']))

    df = NTC_H2_tot_dir

    is_ammonia = df['zone 1'].str.contains('Ammonia', case=False, na=False) | df['zone 2'].str.contains('Ammonia', case=False, na=False)
    is_bottleneck = df.set_index(['zone 1', 'zone 2']).index.isin(bottleneck_pairs)

    valid_for_exp_imp = ~is_ammonia & ~is_bottleneck

    export_NTC_H2_df = df[
        df['zone 1'].isin(h2_zones_set) &
        ~df['zone 2'].isin(h2_zones_set) &
        valid_for_exp_imp
    ].copy()

    import_NTC_H2_df = df[
        ~df['zone 1'].isin(h2_zones_set) &
        df['zone 2'].isin(h2_zones_set) &
        valid_for_exp_imp
    ].copy()

    print(f"✅ Defined export_NTC_H2_df ({len(export_NTC_H2_df)}) and import_NTC_H2_df ({len(import_NTC_H2_df)}).")
else:
    print("⚠️ Required variables are missing.")


✅ Defined export_NTC_H2_df (17) and import_NTC_H2_df (17).


In [ ]:
if 'export_NTC_H2_df' in locals() and 'import_NTC_H2_df' in locals():
    exp_zones = set(export_NTC_H2_df['zone 1']).union(set(export_NTC_H2_df['zone 2']))
    imp_zones = set(import_NTC_H2_df['zone 1']).union(set(import_NTC_H2_df['zone 2']))
    all_involved_zones = exp_zones.union(imp_zones)

    not_in_h2 = sorted(list(all_involved_zones - set(H2_zones)))

    print(f"Found {len(not_in_h2)} zones in export/import NTCs not collected in H2_zones:")
    print(not_in_h2)
else:
    print("❌ export_NTC_H2_df or import_NTC_H2_df is not available.")


Found 8 zones in export/import NTCs not collected in H2_zones:
['DZ', 'IL', 'MA', 'NO', 'TN', 'TR', 'UA', 'Y_NO']


In [ ]:
if 'export_NTC_H2_df' in locals() and 'import_NTC_H2_df' in locals() and 'not_in_h2' in locals():
    import pandas as pd

    # Combine export and import DataFrames
    combined_exp_imp = pd.concat([export_NTC_H2_df, import_NTC_H2_df])

    # Filter for connections involving the 'not_in_h2' zones
    connected_pairs = combined_exp_imp[
        combined_exp_imp['zone 1'].isin(not_in_h2) | combined_exp_imp['zone 2'].isin(not_in_h2)
    ]

    # Display the relevant connections
    print("Connections between H2_zones and not_in_h2 zones:")
    display(connected_pairs[['Border', 'zone 1', 'zone 2', 'capacity, MW']])

    # Get unique H2 zones connected to them
    connected_h2_zones = set()
    for _, row in connected_pairs.iterrows():
        z1, z2 = row['zone 1'], row['zone 2']
        if z1 in H2_zones:
            connected_h2_zones.add(z1)
        if z2 in H2_zones:
            connected_h2_zones.add(z2)

    print(f"\nUnique H2_zones connected to external zones: {sorted(list(connected_h2_zones))}")
else:
    print("❌ Required variables (export_NTC_H2_df, import_NTC_H2_df, not_in_h2) are not available.")


Connections between H2_zones and not_in_h2 zones:


,Border,zone 1,zone 2,"capacity, MW"
14,BEh2-NO,BEh2,NO,0.000
68,DEh2-Y_NO,DEh2,Y_NO,0.000
70,DKh2-NO,DKh2,NO,0.000
77,DZ-ESh2,ESh2,DZ,0.000
79,DZ-ITh2,ITh2,DZ,0.000
90,ESh2-MA,ESh2,MA,0.000
106,FRh2-NO,FRh2,NO,0.000
124,GRh2-IL,GRh2,IL,0.000
128,GRh2-TR,GRh2,TR,0.000
144,HUh2-UA,HUh2,UA,0.000



Unique H2_zones connected to external zones: ['BEh2', 'DEh2', 'DKh2', 'ESh2', 'FRh2', 'GRh2', 'HUh2', 'ITh2', 'NLh2', 'PLh2', 'ROh2', 'UKh2']


In [ ]:
if 'connected_pairs' in locals() and 'not_in_h2' in locals() and 'H2_zones' in locals():
    external_to_h2_mapping = {}

    for ext_zone in not_in_h2:
        # Filter connections involving this specific external zone
        ext_conns = connected_pairs[
            (connected_pairs['zone 1'] == ext_zone) |
            (connected_pairs['zone 2'] == ext_zone)
        ]

        # Find which of the connected zones are in H2_zones
        h2_links = set()
        for _, row in ext_conns.iterrows():
            z1, z2 = row['zone 1'], row['zone 2']
            if z1 in H2_zones:
                h2_links.add(z1)
            if z2 in H2_zones:
                h2_links.add(z2)

        external_to_h2_mapping[ext_zone] = sorted(list(h2_links))

    print("--- Mapping of External Zones to H2 Zones ---")
    for ext_zone, h2_list in external_to_h2_mapping.items():
        print(f"{ext_zone} is connected to: {', '.join(h2_list)}")
else:
    print("❌ Required variables (connected_pairs, not_in_h2, H2_zones) are missing.")

--- Mapping of External Zones to H2 Zones ---
DZ is connected to: ESh2, ITh2
IL is connected to: GRh2
MA is connected to: ESh2
NO is connected to: BEh2, DEh2, DKh2, FRh2, NLh2, UKh2
TN is connected to: ITh2
TR is connected to: GRh2, ITh2
UA is connected to: HUh2, PLh2, ROh2
Y_NO is connected to: DEh2


In [ ]:
external_to_h2_mapping

{'DZ': ['ESh2', 'ITh2'],
 'IL': ['GRh2'],
 'MA': ['ESh2'],
 'NO': ['BEh2', 'DEh2', 'DKh2', 'FRh2', 'NLh2', 'UKh2'],
 'TN': ['ITh2'],
 'TR': ['GRh2', 'ITh2'],
 'UA': ['HUh2', 'PLh2', 'ROh2'],
 'Y_NO': ['DEh2']}

In [ ]:
import_NTC_H2_df

,Border,zone 1,zone 2,"capacity, MW",losses
15,BEh2-NO,NO,BEh2,0.000,1.0
69,DEh2-Y_NO,Y_NO,DEh2,0.000,1.0
71,DKh2-NO,NO,DKh2,0.000,1.0
76,DZ-ESh2,DZ,ESh2,0.000,1.0
78,DZ-ITh2,DZ,ITh2,15819.209,1.0
91,ESh2-MA,MA,ESh2,0.000,1.0
107,FRh2-NO,NO,FRh2,0.000,1.0
125,GRh2-IL,IL,GRh2,0.000,1.0
129,GRh2-TR,TR,GRh2,0.000,1.0
145,HUh2-UA,UA,HUh2,0.000,1.0


## External_external H2

In [ ]:
import pandas as pd

if 'NTC_H2_tot_dir' in locals() and 'H2_zones' in globals() and 'Bottlenecks_H2_df' in globals():
    h2_zones_set = set(H2_zones)
    bottleneck_pairs = set(zip(Bottlenecks_H2_df['zone 1'], Bottlenecks_H2_df['zone 2']))

    df = NTC_H2_tot_dir

    is_ammonia = df['zone 1'].str.contains('Ammonia', case=False, na=False) | df['zone 2'].str.contains('Ammonia', case=False, na=False)
    is_bottleneck = df.set_index(['zone 1', 'zone 2']).index.isin(bottleneck_pairs)

    ext_ext_H2_df = df[
        (~df['zone 1'].isin(h2_zones_set)) &
        (~df['zone 2'].isin(h2_zones_set)) &
        ~is_ammonia &
        ~is_bottleneck
    ].copy()

    external_external_NTCs_H2_df = ext_ext_H2_df
    print(f"✅ Defined external_external_NTCs_H2_df with {len(external_external_NTCs_H2_df)} records.")
else:
    print("❌ Required dataframes or variables not found.")


✅ Defined external_external_NTCs_H2_df with 4 records.


## Offshore H2

In [ ]:
H2_offshore_list = ['DKB2_OFF','DKHE_OFF','DKK2_OFF','DE_OFF','NL0B_OFF','NL0C_OFF','NL0D_OFF','NL0E_OFF','NL0F_OFF',
               'NL0G_OFF','NL0J_OFF','NL0K_OFF','NL0L_OFF','NL0M_OFF','NL0N_OFF','NL0P_OFF','NL0Q_OFF','NL0R_OFF',
               'NL0S_OFF','NL0T_OFF','NL0U_OFF','NL0V_OFF','NL0W_OFF','NL0X_OFF','NL0Y_OFF'] # 'NLLL_OFF'

In [ ]:
# Step 1: Initialize the Offshore DataFrame
import pandas as pd
NTC_H2_offshore_df = pd.DataFrame({'zone 1': H2_offshore_list})
NTC_H2_offshore_df['zone 2'] = 'Offshore_Pool'
NTC_H2_offshore_df['capacity, MW'] = 1000
NTC_H2_offshore_df['losses'] = 1.0

In [ ]:
# Step 2: Define and apply the Onshore mapping
offshore_mapping = {
    "DKh2": ['DKB2_OFF','DKHE_OFF','DKK2_OFF'],
    "DEh2": ['DE_OFF'],
    "NLh2":['NL0B_OFF','NL0C_OFF','NL0D_OFF','NL0E_OFF','NL0F_OFF','NL0G_OFF','NL0J_OFF',
            'NL0K_OFF','NL0L_OFF','NL0M_OFF','NL0N_OFF','NL0P_OFF','NL0Q_OFF','NL0R_OFF',
            'NL0S_OFF','NL0T_OFF','NL0U_OFF','NL0V_OFF','NL0W_OFF','NL0X_OFF','NL0Y_OFF']
    #"NLLL_OFF":['NLLL_OFF'],
}

offshore_to_onshore = {node: onshore for onshore, nodes in offshore_mapping.items() for node in nodes}

# Update zone 2 using the mapping
NTC_H2_offshore_df['zone 2'] = NTC_H2_offshore_df['zone 1'].map(offshore_to_onshore).fillna('Offshore_Pool')

In [ ]:
import pandas as pd

# The user pointed out that H2_offshore_list (25 nodes) is a subset of the
# 44 offshore nodes identified in the electricity sector (profile_list).

# Verification of the overlap
if 'H2_offshore_list' in locals() and 'network_offshore_zones' in locals():
    h2_off_set = set(H2_offshore_list)
    common_nodes = h2_off_set.intersection(network_offshore_zones)

    print(f"Verification: {len(common_nodes)} out of {len(h2_off_set)} H2 offshore nodes are present in the global network topology.")

# Ensure zone 2 in NTC_H2_offshore_df is mapped to the onshore project zones
if 'NTC_H2_offshore_df' in locals():
    # Update zone 2 using the mapping to onshore zones (DEh2, DKh2, NLh2)
    # This is already handled in previous logic, but let's display the final state
    print("--- Hydrogen Offshore-to-Onshore Mapping ---")
    display(NTC_H2_offshore_df.groupby('zone 2')['zone 1'].count().reset_index(name='Connection Count'))

    # Check for any 'Offshore_Pool' remainders
    unmapped = NTC_H2_offshore_df[NTC_H2_offshore_df['zone 2'] == 'Offshore_Pool']
    if not unmapped.empty:
        print(f"Warning: {len(unmapped)} nodes mapped to Offshore_Pool:")
        print(unmapped['zone 1'].tolist())
    else:
        print("✅ All H2 offshore nodes successfully mapped to project onshore zones.")

Verification: 23 out of 25 H2 offshore nodes are present in the global network topology.
--- Hydrogen Offshore-to-Onshore Mapping ---


,zone 2,Connection Count
0,DEh2,1
1,DKh2,3
2,NLh2,21


✅ All H2 offshore nodes successfully mapped to project onshore zones.


## NTCs - Summary

In [ ]:
print(f"Internal: {len(internal_NTC_H2)}")
print(f"Bottlenecks: {len(Bottlenecks_H2_df)}")
print(f"Export: {len(export_NTC_H2_df)}")
print(f"Import: {len(import_NTC_H2_df)}")
print(f"External-External: {len(ext_ext_H2_df)}")
print(f"Ammonia: {len(Ammonia_H2_df)}")
print("-" * 30)

calculated_sum = (
    len(internal_NTC_H2) +
    len(Bottlenecks_H2_df) +
    len(export_NTC_H2_df) +
    len(import_NTC_H2_df) +
    len(ext_ext_H2_df) +
    len(Ammonia_H2_df)
)

original_len = len(NTC_H2_tot_dir)

print(f"Sum of parts: {calculated_sum}")
print(f"Original length (NTC_H2_tot_dir): {original_len}")
print(f"Matches Original: {calculated_sum == original_len}")

Internal: 122
Bottlenecks: 38
Export: 17
Import: 17
External-External: 4
Ammonia: 13
------------------------------
Sum of parts: 211
Original length (NTC_H2_tot_dir): 211
Matches Original: True


# Project  NTCs - Filter

In [ ]:
import json

# 1. Define the mapping between electricity zones and hydrogen zones
zone_mapping = {}
assigned_h2_zones = set()

# Prioritize ITN1 for ITh2 mapping as per project requirements
for zone in zones:
    prefix = zone[:2]
    matched_h2 = [h2 for h2 in H2_zones if h2.startswith(prefix)]

    # Special logic: Italy zones (except ITN1) should not take ITh2 yet
    if prefix == 'IT' and zone != 'ITN1':
        matched_h2 = [h2 for h2 in matched_h2 if h2 != 'ITh2']

    # Manual injection for ITN1 if it somehow wasn't in the prefix match
    if zone == 'ITN1' and 'ITh2' not in assigned_h2_zones:
        if 'ITh2' not in matched_h2: matched_h2.append('ITh2')

    final_h2_for_zone = []
    for h2 in matched_h2:
        if h2 not in assigned_h2_zones:
            final_h2_for_zone.append(h2)
            assigned_h2_zones.add(h2)

    zone_mapping[zone] = final_h2_for_zone

print("--- Final Hydrogen Zone Mapping ---")
for z, h2_list in zone_mapping.items():
    print(f"{z}: {h2_list}")

--- Final Hydrogen Zone Mapping ---
AL00: ['ALh2']
AT00: ['ATh2']
BA00: ['BAh2']
BE00: ['BEh2']
BG00: ['BGh2']
CH00: ['CHh2']
CY00: ['CYh2']
CZ00: ['CZh2']
DE00: ['DEh2', 'DEh2Z1']
DKE1: ['DKh2']
DKW1: []
EE00: ['EEh2']
ES00: ['ESh2']
FI00: ['FIh2', 'FIh2Al', 'FIh2N', 'FIh2S']
FR00: ['FRh2', 'FRh2N', 'FRh2S', 'FRh2SW']
GR00: ['GRh2']
GR03: []
HR00: ['HRh2']
HU00: ['HUh2']
IE00: ['IEh2']
ITCA: []
ITCN: []
ITCS: []
ITN1: ['ITh2']
ITS1: []
ITSA: []
ITSI: []
LT00: ['LTh2', 'LTh2Z1']
LUF1: ['LUh2']
LUG1: []
LUV1: []
LV00: ['LVh2']
MD00: ['MDh2']
ME00: []
MK00: ['MKh2']
MT00: ['MTh2']
NL00: ['NLh2']
NOM1: ['NOh2']
NON1: []
NOS1: []
NOS2: []
NOS3: []
PL00: ['PLh2']
PT00: ['PTh2', 'PTh2Z1']
RO00: ['ROh2']
RS00: ['RSh2']
SE01: ['SEh2']
SE02: []
SE03: []
SE04: []
SI00: ['SIh2']
SK00: ['SKh2E', 'SKh2W']
TR00: []
UA00: []
UK00: ['UKh2']
UKNI: []


In [ ]:
# Create the H2_project_zone list based on the previous mapping
H2_project_zone = []

for elec_zone, h2_mapped_list in zone_mapping.items():
    for h2_zone in h2_mapped_list:
        if h2_zone not in H2_project_zone:
            H2_project_zone.append(h2_zone)

print(f"H2 project zones: {H2_project_zone}")

H2 project zones: ['ALh2', 'ATh2', 'BAh2', 'BEh2', 'BGh2', 'CHh2', 'CYh2', 'CZh2', 'DEh2', 'DEh2Z1', 'DKh2', 'EEh2', 'ESh2', 'FIh2', 'FIh2Al', 'FIh2N', 'FIh2S', 'FRh2', 'FRh2N', 'FRh2S', 'FRh2SW', 'GRh2', 'HRh2', 'HUh2', 'IEh2', 'ITh2', 'LTh2', 'LTh2Z1', 'LUh2', 'LVh2', 'MDh2', 'MKh2', 'MTh2', 'NLh2', 'NOh2', 'PLh2', 'PTh2', 'PTh2Z1', 'ROh2', 'RSh2', 'SEh2', 'SIh2', 'SKh2E', 'SKh2W', 'UKh2']


In [ ]:
H2_Bottlenecks_mapping = {
"ITN1" : ['IB_ITh2'],
"SK00" : ['IB_SKh2C', 'IB_SKh2E'],
"PL00" : ['PLh2nbc'],
"UK00" : ['UKh2/INT'],
"GR00" : ['IB_GRh2P']
}

In [ ]:
import pandas as pd

# 1. Identify relevant bottleneck nodes from the mapping for current project regions
project_bottleneck_nodes = []
if 'H2_Bottlenecks_mapping' in locals() and 'zones' in locals():
    print("--- Bottleneck Mapping Check ---")
    for elec_zone, b_list in H2_Bottlenecks_mapping.items():
        if elec_zone in zones:
            print(f"✅ Zone {elec_zone} is in project. Adding bottlenecks: {b_list}")
            for b_node in b_list:
                if b_node not in project_bottleneck_nodes:
                    project_bottleneck_nodes.append(b_node)

# 2. Filter the global Bottlenecks_H2_df for these specific nodes
if 'Bottlenecks_H2_df' in locals() and project_bottleneck_nodes:
    NTC_h2_bottlenecks_project_df = Bottlenecks_H2_df[
        (Bottlenecks_H2_df['zone 1'].isin(project_bottleneck_nodes)) |
        (Bottlenecks_H2_df['zone 2'].isin(project_bottleneck_nodes))
    ].copy()

    print(f"\n✅ Created NTC_h2_bottlenecks_project_df with {len(NTC_h2_bottlenecks_project_df)} connections.")
    display(NTC_h2_bottlenecks_project_df)
else:
    print("\nℹ️ No bottlenecks found for the current project zones.")
    NTC_h2_bottlenecks_project_df = pd.DataFrame(columns=['Border', 'zone 1', 'zone 2', 'capacity, MW', 'losses'])


--- Bottleneck Mapping Check ---
✅ Zone ITN1 is in project. Adding bottlenecks: ['IB_ITh2']
✅ Zone SK00 is in project. Adding bottlenecks: ['IB_SKh2C', 'IB_SKh2E']
✅ Zone PL00 is in project. Adding bottlenecks: ['PLh2nbc']
✅ Zone UK00 is in project. Adding bottlenecks: ['UKh2/INT']
✅ Zone GR00 is in project. Adding bottlenecks: ['IB_GRh2P']

✅ Created NTC_h2_bottlenecks_project_df with 32 connections.


,Border,zone 1,zone 2,"capacity, MW",losses
8,ATh2-IB_ITh2,ATh2,IB_ITh2,4449.152542,1.0
9,ATh2-IB_ITh2,IB_ITh2,ATh2,5932.203392,1.0
30,BEh2-UKh2/INT,BEh2,UKh2/INT,0.000000,1.0
31,BEh2-UKh2/INT,UKh2/INT,BEh2,0.000000,1.0
42,CHh2-IB_ITh2,CHh2,IB_ITh2,0.000000,1.0
43,CHh2-IB_ITh2,IB_ITh2,CHh2,0.000000,1.0
66,DEh2-PLh2nbc,DEh2,PLh2nbc,3531.073446,1.0
67,DEh2-PLh2nbc,PLh2nbc,DEh2,7062.146892,1.0
126,IB_GRh2P-ITh2,IB_GRh2P,ITh2,0.000000,1.0
127,IB_GRh2P-ITh2,ITh2,IB_GRh2P,0.000000,1.0


In [ ]:
import pandas as pd

# Standardize matching by using the full project list
h2_proj_set = set(H2_project_zone)
project_bottleneck_set = set(project_bottleneck_nodes)

# 1. Project-Specific Internal H2 NTCs
NTC_h2_internal_project_df = internal_NTC_H2[
    internal_NTC_H2['zone 1'].isin(h2_proj_set) &
    internal_NTC_H2['zone 2'].isin(h2_proj_set)
].copy()

# 2. Project-Specific Ammonia H2 NTCs
NTC_h2_ammonia_project_df = Ammonia_H2_df[
    Ammonia_H2_df['zone 2'].isin(h2_proj_set) |
    Ammonia_H2_df['zone 2'].isin(project_bottleneck_set)
].copy()

# 3. Project-Specific Import H2 NTCs
NTC_h2_import_project_df = import_NTC_H2_df[
    import_NTC_H2_df['zone 2'].isin(h2_proj_set)
].copy()

# 4. Project-Specific Export H2 NTCs
NTC_h2_export_project_df = export_NTC_H2_df[
    export_NTC_H2_df['zone 1'].isin(h2_proj_set)
].copy()

# 5. Project-Specific External-External H2 NTCs
h2_neighbors = set(NTC_h2_import_project_df['zone 1']) | set(NTC_h2_export_project_df['zone 2'])
NTC_h2_ext_ext_project_df = external_external_NTCs_H2_df[
    (external_external_NTCs_H2_df['zone 1'].isin(h2_neighbors)) &
    (external_external_NTCs_H2_df['zone 2'].isin(h2_neighbors))
].copy()



# Update dictionary
NTC_h2_dic = {
    'internal': NTC_h2_internal_project_df,
    'ammonia': NTC_h2_ammonia_project_df,
    'bottlenecks': NTC_h2_bottlenecks_project_df,
    'import': NTC_h2_import_project_df,
    'export': NTC_h2_export_project_df,
    'ext_ext': NTC_h2_ext_ext_project_df,
    'offshore': NTC_H2_offshore_df
}

print("--- Updated Project Hydrogen Counts ---")
for k, v in NTC_h2_dic.items():
    print(f"{k.capitalize()}: {len(v)}")

# 6. Final Consolidation
h2_list = []
for cat, df in NTC_h2_dic.items():
    temp = df.copy()
    temp['Sector'] = 'Hydrogen'
    temp['Category'] = cat.replace('_', ' ').capitalize()
    h2_list.append(temp)

h2_summary = pd.concat(h2_list, ignore_index=True)


--- Updated Project Hydrogen Counts ---
Internal: 122
Ammonia: 13
Bottlenecks: 32
Import: 17
Export: 17
Ext_ext: 4
Offshore: 25


In [ ]:
import json
import os
import pandas as pd

# Refresh the electricity dictionary with the filtered offshore data
NTC_elec_dic = {
    'internal': NTC_elec_project_df,
    'import': import_NTCs_project_df,
    'export': export_NTCs_project_df,
    'offshore_direct': offshore_direct_project_df,
    'offshore_offshore': offshore_offshore_project_df,
    'virtual': virtual_NTC_project_df,
    'ext_ext': external_external_project_df,
    'offshore_hub': offshore_hubs_project
}

# Combine Electricity and Hydrogen dictionaries into one master NTC_dic
NTC_dic = {
    'Electricity': NTC_elec_dic,
    'Hydrogen': NTC_h2_dic
}

def serialize_ntc_dic(d):
    if isinstance(d, dict):
        return {k: serialize_ntc_dic(v) for k, v in d.items()}
    elif isinstance(d, pd.DataFrame):
        return d.to_dict(orient='records')
    return d

NTC_dic_serializable = serialize_ntc_dic(NTC_dic)
ntc_save_path = os.path.join(INTER_DIR, 'NTC_dic.json')
os.makedirs(INTER_DIR, exist_ok=True)

try:
    with open(ntc_save_path, 'w', encoding='utf-8') as f:
        json.dump(NTC_dic_serializable, f, ensure_ascii=False, indent=4)
    print(f"✅ Master NTC_dic successfully updated and saved to: {ntc_save_path}")

    print("\nUpdated Electricity category counts:")
    for cat, data in NTC_elec_dic.items():
        count = len(data)
        print(f"- {cat.replace('_', ' ').capitalize()}: {count} items")

except Exception as e:
    print(f"❌ Failed to save combined NTC_dic: {e}")

✅ Master NTC_dic successfully updated and saved to: /content/drive/MyDrive/Colab_Notebooks/ENNOH/Zonal_model/TYNDP_scenario_2026/intermediate_data/Europe/DE/2035/NTC_dic.json

Updated Electricity category counts:
- Internal: 226 items
- Import: 9 items
- Export: 9 items
- Offshore direct: 84 items
- Offshore offshore: 18 items
- Virtual: 28 items
- Ext ext: 4 items
- Offshore hub: 6 items


# Offshore Profiles

In [ ]:
if 'NTC_tot_df' in locals() and 'profile_list' in locals():
    # Identify connections where one zone is onshore and the other is in the offshore profile list
    onshore_offshore_mapping = []

    for _, row in NTC_tot_df.iterrows():
        z1, z2 = row['zone 1'], row['zone 2']

        # Check if one side is in project zones and the other is a known offshore profile node
        is_proj_z1 = z1 in zones
        is_proj_z2 = z2 in zones
        is_prof_z1 = z1 in profile_list
        is_prof_z2 = z2 in profile_list

        if (is_proj_z1 and is_prof_z2) or (is_proj_z2 and is_prof_z1):
            onshore_offshore_mapping.append(row)

    onshore_offshore_mapping_df = pd.DataFrame(onshore_offshore_mapping).reset_index(drop=True)

    print(f"✅ Found {len(onshore_offshore_mapping_df)} project-specific connections to offshore profiles.")
    if not onshore_offshore_mapping_df.empty:
        display(onshore_offshore_mapping_df)
    else:
        print("ℹဴ No direct connections found between current project zones and the 54 offshore profiles.")
else:
    print("❌ Required variables (NTC_tot_df or profile_list) are missing.")

✅ Found 84 project-specific connections to offshore profiles.


,zone 1,zone 2,"capacity, MW",losses
0,BE00,BEO1_OFF,1400.0,0.95
1,BEO1_OFF,BE00,1400.0,0.95
2,BE00,BEO2_OFF,0.0,0.95
3,BEO2_OFF,BE00,0.0,0.95
4,BE00,DKNS_OFF,0.0,0.95
...,...,...,...,...
79,NOM1,NONC_OFF,0.0,0.95
80,NOND_OFF,NON1,0.0,0.95
81,NON1,NOND_OFF,0.0,0.95
82,NOWB_OFF,NOS3,0.0,0.95


In [ ]:
if 'NTC_tot_df' in locals():
    # Filter NTC_tot_df for any connections involving 'ES00'
    es00_ntc_df = NTC_tot_df[(NTC_tot_df['zone 1'] == 'ES00') | (NTC_tot_df['zone 2'] == 'ES00')].copy()

    print(f"Found {len(es00_ntc_df)} NTC connections involving ES00.")
    display(es00_ntc_df)
else:
    print("❌ NTC_tot_df is not defined. Please ensure the electricity NTC data has been loaded.")

Found 10 NTC connections involving ES00.


,zone 1,zone 2,"capacity, MW",losses
116,ES00,FR00,5000.0,0.95
117,FR00,ES00,5000.0,0.95
118,ES00,MA00,900.0,0.95
119,MA00,ES00,600.0,0.95
120,ES00,PT00,4200.0,0.95
121,PT00,ES00,3500.0,0.95
320,ES00,ESC1_OFF,300.0,0.95
321,ESC1_OFF,ES00,300.0,0.95
322,ES00,ESG2_OFF,200.0,0.95
323,ESG2_OFF,ES00,200.0,0.95


In [ ]:
if 'NTC_tot_df' in locals():
    # Filter for connections involving ESC1_OFF or ESG2_OFF
    target_offshore_nodes = ['ESC1_OFF', 'ESG2_OFF']
    offshore_ntc_filtered = NTC_tot_df[
        (NTC_tot_df['zone 1'].isin(target_offshore_nodes)) |
        (NTC_tot_df['zone 2'].isin(target_offshore_nodes))
    ].copy()

    print(f"Found {len(offshore_ntc_filtered)} connections involving ESC1_OFF or ESG2_OFF.")
    display(offshore_ntc_filtered)
else:
    print("❌ NTC_tot_df is not defined.")

Found 4 connections involving ESC1_OFF or ESG2_OFF.


,zone 1,zone 2,"capacity, MW",losses
320,ES00,ESC1_OFF,300.0,0.95
321,ESC1_OFF,ES00,300.0,0.95
322,ES00,ESG2_OFF,200.0,0.95
323,ESG2_OFF,ES00,200.0,0.95


In [ ]:
if 'NTC_tot_df' in locals():
    # 1. Filter all connections involving at least one 'OFF' node
    offshore_mask = NTC_tot_df['zone 1'].str.contains('OFF', na=False) | NTC_tot_df['zone 2'].str.contains('OFF', na=False)
    all_off_conns = NTC_tot_df[offshore_mask].copy()

    # 2. Map every offshore node to its set of unique neighbors
    offshore_hubs_check = {}

    for _, row in all_off_conns.iterrows():
        z1, z2 = row['zone 1'], row['zone 2']

        if 'OFF' in z1:
            offshore_hubs_check.setdefault(z1, set()).add(z2)
        if 'OFF' in z2:
            offshore_hubs_check.setdefault(z2, set()).add(z1)

    # 3. Filter for nodes with > 1 unique neighbor
    potential_hubs = [
        {'Offshore Node': node, 'Neighbors': sorted(list(neighbors)), 'Connection Count': len(neighbors)}
        for node, neighbors in offshore_hubs_check.items() if len(neighbors) > 1
    ]

    hub_df = pd.DataFrame(potential_hubs).sort_values(by='Connection Count', ascending=False)

    print(f"Found {len(hub_df)} offshore nodes connected to more than one zone (Hubs):")
    display(hub_df)
else:
    print("❌ NTC_tot_df is not defined.")

Found 14 offshore nodes connected to more than one zone (Hubs):


,Offshore Node,Neighbors,Connection Count
1,DKNS_OFF,"[BE00, DE00, DKW1]",3
3,DKBH_OFF,"[DE00, DKB2_OFF, DKE1]",3
0,BEO1_OFF,"[BE00, UK00]",2
2,DEKF_OFF,"[DE00, DKKF_OFF]",2
4,NOSF_OFF,"[DE00, NOS2]",2
5,DKKF_OFF,"[DEKF_OFF, DKE1]",2
6,NL0C_OFF,"[NL00, NL0G_OFF]",2
7,NL0D_OFF,"[NL00, NL0E_OFF]",2
8,NL0J_OFF,"[NL00, NL0K_OFF]",2
9,NL0L_OFF,"[NL00, NL0M_OFF]",2


In [ ]:
import pandas as pd

if 'Bottlenecks_H2_df' in locals():
    # Filter the global bottlenecks dataframe for 'PLh2nbc'
    plh2nbc_ntcs = Bottlenecks_H2_df[
        (Bottlenecks_H2_df['zone 1'].str.contains('PLh2nbc', na=False)) |
        (Bottlenecks_H2_df['zone 2'].str.contains('PLh2nbc', na=False)) |
        (Bottlenecks_H2_df['Border'].str.contains('PLh2nbc', na=False))
    ]

    if not plh2nbc_ntcs.empty:
        print(f"✅ Found {len(plh2nbc_ntcs)} NTCs using PLh2nbc bottlenecks in the global dataset:")
        display(plh2nbc_ntcs)
    else:
        print("ℹ️ No NTCs found using the PLh2nbc bottlenecks in the global dataframe.")

    # Check in the project-filtered dataframe to see if it was included for this specific project
    if 'NTC_h2_bottlenecks_project_df' in locals():
        proj_pl = NTC_h2_bottlenecks_project_df[
            (NTC_h2_bottlenecks_project_df['zone 1'].str.contains('PLh2nbc', na=False)) |
            (NTC_h2_bottlenecks_project_df['zone 2'].str.contains('PLh2nbc', na=False))
        ]
        if not proj_pl.empty:
            print(f"\n✅ In the project-specific bottlenecks dataframe: {len(proj_pl)} connections found.")
        else:
            print("\nℹ️ Note: 'PLh2nbc' is NOT included in the project-specific filtered bottlenecks (NTC_h2_bottlenecks_project_df).")
else:
    print("❌ Bottlenecks_H2_df is not available. Please ensure previous cells were run.")

✅ Found 6 NTCs using PLh2nbc bottlenecks in the global dataset:


,Border,zone 1,zone 2,"capacity, MW",losses
66,DEh2-PLh2nbc,DEh2,PLh2nbc,3531.073446,1.0
67,DEh2-PLh2nbc,PLh2nbc,DEh2,7062.146892,1.0
184,LTh2-PLh2nbc,LTh2,PLh2nbc,7062.146892,1.0
185,LTh2-PLh2nbc,PLh2nbc,LTh2,3531.073446,1.0
194,PLh2-PLh2nbc,PLh2,PLh2nbc,1765.536708,1.0
195,PLh2-PLh2nbc,PLh2nbc,PLh2,3531.073417,1.0



✅ In the project-specific bottlenecks dataframe: 6 connections found.


In [ ]:
print("Global Hydrogen Bottleneck NTCs:")
display(Bottlenecks_H2_df)

print("\nProject-Specific Hydrogen Bottleneck NTCs:")
display(NTC_h2_bottlenecks_project_df)

Global Hydrogen Bottleneck NTCs:


,Border,zone 1,zone 2,"capacity, MW",losses
8,ATh2-IB_ITh2,ATh2,IB_ITh2,4449.152542,1.0
9,ATh2-IB_ITh2,IB_ITh2,ATh2,5932.203392,1.0
30,BEh2-UKh2/INT,BEh2,UKh2/INT,0.000000,1.0
31,BEh2-UKh2/INT,UKh2/INT,BEh2,0.000000,1.0
42,CHh2-IB_ITh2,CHh2,IB_ITh2,0.000000,1.0
43,CHh2-IB_ITh2,IB_ITh2,CHh2,0.000000,1.0
48,CZh2-IB_SKh2W,CZh2,IB_SKh2W,0.000000,1.0
49,CZh2-IB_SKh2W,IB_SKh2W,CZh2,5084.745750,1.0
66,DEh2-PLh2nbc,DEh2,PLh2nbc,3531.073446,1.0
67,DEh2-PLh2nbc,PLh2nbc,DEh2,7062.146892,1.0



Project-Specific Hydrogen Bottleneck NTCs:


,Border,zone 1,zone 2,"capacity, MW",losses
8,ATh2-IB_ITh2,ATh2,IB_ITh2,4449.152542,1.0
9,ATh2-IB_ITh2,IB_ITh2,ATh2,5932.203392,1.0
30,BEh2-UKh2/INT,BEh2,UKh2/INT,0.000000,1.0
31,BEh2-UKh2/INT,UKh2/INT,BEh2,0.000000,1.0
42,CHh2-IB_ITh2,CHh2,IB_ITh2,0.000000,1.0
43,CHh2-IB_ITh2,IB_ITh2,CHh2,0.000000,1.0
66,DEh2-PLh2nbc,DEh2,PLh2nbc,3531.073446,1.0
67,DEh2-PLh2nbc,PLh2nbc,DEh2,7062.146892,1.0
126,IB_GRh2P-ITh2,IB_GRh2P,ITh2,0.000000,1.0
127,IB_GRh2P-ITh2,ITh2,IB_GRh2P,0.000000,1.0


In [ ]:
import pandas as pd

print("--- Electricity NTCs involving UA ---")
if 'NTC_tot_df' in locals():
    ua_elec_ntcs = NTC_tot_df[
        (NTC_tot_df['zone 1'].str.contains('UA', na=False)) |
        (NTC_tot_df['zone 2'].str.contains('UA', na=False))
    ]
    display(ua_elec_ntcs)
else:
    print("NTC_tot_df is not available.")

print("\n--- Hydrogen NTCs involving UA ---")
if 'NTC_H2_tot_dir' in locals():
    ua_h2_ntcs = NTC_H2_tot_dir[
        (NTC_H2_tot_dir['zone 1'].str.contains('UA', na=False)) |
        (NTC_H2_tot_dir['zone 2'].str.contains('UA', na=False))
    ]
    display(ua_h2_ntcs)
else:
    print("NTC_H2_tot_dir is not available.")


--- Electricity NTCs involving UA ---


,zone 1,zone 2,"capacity, MW",losses
274,HU00,UA00,840.0,0.95
275,UA00,HU00,220.0,0.95
276,PL00,UA00,500.0,0.95
277,UA00,PL00,500.0,0.95
278,RO00,UA00,300.0,0.95
279,UA00,RO00,50.0,0.95
280,SK00,UA00,400.0,0.95
281,UA00,SK00,400.0,0.95
282,MD00,UA00,600.0,0.95
283,UA00,MD00,600.0,0.95



--- Hydrogen NTCs involving UA ---


,Border,zone 1,zone 2,"capacity, MW",losses
144,HUh2-UA,HUh2,UA,0.000000,1.0
145,HUh2-UA,UA,HUh2,0.000000,1.0
158,IB_SKh2E-UA,IB_SKh2E,UA,0.000000,1.0
159,IB_SKh2E-UA,UA,IB_SKh2E,5084.745762,1.0
196,PLh2-UA,PLh2,UA,0.000000,1.0
197,PLh2-UA,UA,PLh2,0.000000,1.0
200,ROh2-UA,ROh2,UA,0.000000,1.0
201,ROh2-UA,UA,ROh2,0.000000,1.0


In [ ]:
print("--- Global Electricity Import NTCs ---")
display(import_NTCs_df)

print("\n--- Global Electricity Export NTCs ---")
display(export_NTCs_df)

print("\n--- Global Hydrogen Import NTCs ---")
display(import_NTC_H2_df)

print("\n--- Global Hydrogen Export NTCs ---")
display(export_NTC_H2_df)


--- Global Electricity Import NTCs ---


,zone 1,zone 2,"capacity, MW",losses
0,EG00,CY00,0.0,0.95
1,IL00,CY00,1000.0,0.95
2,EG00,GR00,0.0,0.95
3,MA00,ES00,600.0,0.95
4,LY00,GR00,0.0,0.95
5,LY00,GR03,0.0,0.95
6,IS00,UK00,0.0,0.95
7,TN00,ITSI,600.0,0.95
8,EG00,GR03,0.0,0.95



--- Global Electricity Export NTCs ---


,zone 1,zone 2,"capacity, MW",losses
0,CY00,EG00,0.0,0.95
1,CY00,IL00,1000.0,0.95
2,GR00,EG00,0.0,0.95
3,ES00,MA00,900.0,0.95
4,GR00,LY00,0.0,0.95
5,GR03,LY00,0.0,0.95
6,UK00,IS00,0.0,0.95
7,ITSI,TN00,600.0,0.95
8,GR03,EG00,0.0,0.95



--- Global Hydrogen Import NTCs ---


,Border,zone 1,zone 2,"capacity, MW",losses
15,BEh2-NO,NO,BEh2,0.000,1.0
69,DEh2-Y_NO,Y_NO,DEh2,0.000,1.0
71,DKh2-NO,NO,DKh2,0.000,1.0
76,DZ-ESh2,DZ,ESh2,0.000,1.0
78,DZ-ITh2,DZ,ITh2,15819.209,1.0
91,ESh2-MA,MA,ESh2,0.000,1.0
107,FRh2-NO,NO,FRh2,0.000,1.0
125,GRh2-IL,IL,GRh2,0.000,1.0
129,GRh2-TR,TR,GRh2,0.000,1.0
145,HUh2-UA,UA,HUh2,0.000,1.0



--- Global Hydrogen Export NTCs ---


,Border,zone 1,zone 2,"capacity, MW",losses
14,BEh2-NO,BEh2,NO,0.0,1.0
68,DEh2-Y_NO,DEh2,Y_NO,0.0,1.0
70,DKh2-NO,DKh2,NO,0.0,1.0
77,DZ-ESh2,ESh2,DZ,0.0,1.0
79,DZ-ITh2,ITh2,DZ,0.0,1.0
90,ESh2-MA,ESh2,MA,0.0,1.0
106,FRh2-NO,FRh2,NO,0.0,1.0
124,GRh2-IL,GRh2,IL,0.0,1.0
128,GRh2-TR,GRh2,TR,0.0,1.0
144,HUh2-UA,HUh2,UA,0.0,1.0


In [ ]:
import pandas as pd

# Displaying the summary of the network topology stored in the NTC_dic
print(f"--- Detailed Topology Summary for {project_name} (TYNDP 2026) ---")

for sector, categories in NTC_dic.items():
    print(f"\nSector: {sector}")
    for cat, data in categories.items():
        if isinstance(data, pd.DataFrame):
            count = len(data)
        elif isinstance(data, dict):
            count = len(data)
        else:
            count = 0
        print(f"  - {cat.replace('_', ' ').capitalize()}: {count} connections")

# Show sample of Electricity internal connections
print("\nSample Electricity Internal Connections:")
display(NTC_elec_project_df.head())

# Show sample of Hydrogen internal connections
print("\nSample Hydrogen Internal Connections:")
display(NTC_h2_internal_project_df.head())


--- Detailed Topology Summary for Europe (TYNDP 2026) ---

Sector: Electricity
  - Internal: 226 connections
  - Import: 9 connections
  - Export: 9 connections
  - Offshore direct: 84 connections
  - Offshore offshore: 18 connections
  - Virtual: 28 connections
  - Ext ext: 4 connections
  - Offshore hub: 6 connections

Sector: Hydrogen
  - Internal: 122 connections
  - Ammonia: 13 connections
  - Bottlenecks: 32 connections
  - Import: 17 connections
  - Export: 17 connections
  - Ext ext: 4 connections
  - Offshore: 25 connections

Sample Electricity Internal Connections:


,zone 1,zone 2,"capacity, MW",losses
0,AL00,GR00,610.0,0.95
1,GR00,AL00,1040.0,0.95
2,AL00,ME00,300.0,0.95
3,ME00,AL00,300.0,0.95
4,AL00,MK00,298.0,0.95



Sample Hydrogen Internal Connections:


,Border,zone 1,zone 2,"capacity, MW",losses
0,ALh2-HRh2,ALh2,HRh2,0.000000,1.0
1,ALh2-HRh2,HRh2,ALh2,0.000000,1.0
2,ATh2-CZh2,ATh2,CZh2,0.000000,1.0
3,ATh2-CZh2,CZh2,ATh2,0.000000,1.0
4,ATh2-DEh2,ATh2,DEh2,5296.610171,1.0


In [ ]:
NTC_H2_offshore_df

,zone 1,zone 2,"capacity, MW",losses
0,DKB2_OFF,DKh2,1000,1.0
1,DKHE_OFF,DKh2,1000,1.0
2,DKK2_OFF,DKh2,1000,1.0
3,DE_OFF,DEh2,1000,1.0
4,NL0B_OFF,NLh2,1000,1.0
5,NL0C_OFF,NLh2,1000,1.0
6,NL0D_OFF,NLh2,1000,1.0
7,NL0E_OFF,NLh2,1000,1.0
8,NL0F_OFF,NLh2,1000,1.0
9,NL0G_OFF,NLh2,1000,1.0
